In [0]:
%run "/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/src/task3_enriched_orders"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

In [0]:
def test_stg_orders_exists():
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.stg_orders"), "stg_orders table does not exist"


def test_stg_orders_not_empty():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_orders")
    assert df.count() > 0, "stg_orders is empty"

In [0]:
stg_orders = spark.read.table("az_ci_de_dbs.ecom_dev.stg_orders")

In [0]:
# DQ tests on stg_orders

def test_dq_orders_no_null_ids():
    result = dq_orders(stg_orders)
    for col_name in ["order_id", "customer_id", "product_id"]:
        nulls = result.filter(F.col(col_name).isNull()).count()
        assert nulls == 0, f"Found {nulls} nulls in {col_name} after DQ"


def test_dq_orders_no_blank_ids():
    result = dq_orders(stg_orders)
    for col_name in ["order_id", "customer_id", "product_id"]:
        blanks = result.filter(F.trim(F.col(col_name)) == "").count()
        assert blanks == 0, f"Found {blanks} blank values in {col_name} after DQ"


def test_dq_orders_order_id_format():
    result = dq_orders(stg_orders)
    invalid = result.filter(~F.col("order_id").rlike(r"^[A-Z]{2}-\d{4}-\d{6}$")).count()
    assert invalid == 0, f"Found {invalid} invalid order_ids after DQ"


def test_dq_orders_customer_id_format():
    result = dq_orders(stg_orders)
    invalid = result.filter(~F.col("customer_id").rlike(r"^[A-Z]{2}-\d{5}$")).count()
    assert invalid == 0, f"Found {invalid} invalid customer_ids after DQ"


def test_dq_orders_product_id_format():
    result = dq_orders(stg_orders)
    invalid = result.filter(~F.col("product_id").rlike(r"^[A-Z]{3}-[A-Z]{2}-\d{8}$")).count()
    assert invalid == 0, f"Found {invalid} invalid product_ids after DQ"


def test_dq_orders_price_non_negative():
    result = dq_orders(stg_orders)
    invalid = result.filter(F.col("price") < 0).count()
    assert invalid == 0, f"Found {invalid} negative prices after DQ"


def test_dq_orders_discount_non_negative():
    result = dq_orders(stg_orders)
    invalid = result.filter(F.col("discount") < 0).count()
    assert invalid == 0, f"Found {invalid} negative discounts after DQ"


def test_dq_orders_quantity_positive_integer():
    result = dq_orders(stg_orders)
    invalid = result.filter((F.col("quantity") <= 0) | (F.col("quantity") != F.col("quantity").cast("int"))).count()
    assert invalid == 0, f"Found {invalid} invalid quantities after DQ"


def test_dq_orders_reduces_or_keeps_count():
    result = dq_orders(stg_orders)
    assert result.count() <= stg_orders.count(), "DQ increased row count unexpectedly"

In [0]:
# Dedupe tests

def test_dedupe_orders_no_duplicates():
    df = dq_orders(stg_orders)
    result = dedupe(df)
    total = result.count()
    distinct = result.dropDuplicates().count()
    assert total == distinct, f"Duplicates remain: {total} rows, {distinct} distinct rows"


def test_dedupe_orders_reduces_or_keeps_count():
    df = dq_orders(stg_orders)
    result = dedupe(df)
    assert result.count() <= df.count(), "Dedupe increased row count unexpectedly"


def test_dedupe_exact_duplicate_rows_removed():
    duplicate_rows = [
        (1, "CA-2017-152156", "15/3/2024", "20/3/2024", "Standard", "PW-19240", "FUR-CH-10002961", 2, 100.5, 0.2, 25.68),
        (1, "CA-2017-152156", "15/3/2024", "20/3/2024", "Standard", "PW-19240", "FUR-CH-10002961", 2, 100.5, 0.2, 25.68),
    ]
    duplicate_schema = ["row_id", "order_id", "order_date", "ship_date", "ship_mode", "customer_id", "product_id", "quantity", "price", "discount", "profit"]
    df = spark.createDataFrame(duplicate_rows, duplicate_schema)
    result = dedupe(df)
    assert result.count() == 1, f"Expected 1 row after dedupe, got {result.count()}"

In [0]:
# Tests for build_enriched_orders and main_build_enriched_orders

build_fact_orders(stg_orders)
enriched_df = build_enriched_orders()
enriched_df = dedupe(enriched_df)
delta_writer(enriched_df, "az_ci_de_dbs.ecom_dev.enriched_orders")


def test_dim_tables_exist():
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.dim_product"), "dim_product table does not exist"
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.dim_customer"), "dim_customer table does not exist"


def test_fact_orders_exists():
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.fact_orders"), "fact_orders table does not exist"


def test_build_enriched_orders_not_empty():
    enriched = build_enriched_orders()
    assert enriched.count() > 0, "build_enriched_orders returned empty DataFrame"


def test_build_enriched_orders_expected_columns():
    enriched = build_enriched_orders()
    expected_cols = ["customer_name", "country", "category", "sub_category"]
    for col_name in expected_cols:
        assert col_name in enriched.columns, f"Expected column '{col_name}' missing from enriched result"


def test_build_enriched_orders_no_customer_product_id_cols():
    """After join, customer_id and product_id should not be in the select (they are join keys removed from fact cols)."""
    enriched = build_enriched_orders()
    assert "customer_id" not in enriched.columns, "customer_id should be excluded from enriched output"
    assert "product_id" not in enriched.columns, "product_id should be excluded from enriched output"


def test_build_enriched_orders_retains_fact_columns():
    enriched = build_enriched_orders()
    fact_cols_expected = ["row_id", "order_id", "order_date", "ship_date", "ship_mode", "quantity", "price", "discount", "profit", "order_key"]
    for col_name in fact_cols_expected:
        assert col_name in enriched.columns, f"Fact column '{col_name}' missing from enriched result"


def test_build_enriched_orders_no_nulls_in_joined_cols():
    """Inner join should produce no nulls in the joined dimension columns."""
    enriched = build_enriched_orders()
    for col_name in ["customer_name", "country", "category", "sub_category"]:
        nulls = enriched.filter(F.col(col_name).isNull()).count()
        assert nulls == 0, f"Found {nulls} nulls in {col_name} after enrichment join"


def test_enriched_orders_count_le_fact():
    """Inner join should not produce more rows than fact_orders (assuming no fan-out from dims)."""
    fact_count = spark.read.table("az_ci_de_dbs.ecom_dev.fact_orders").count()
    enriched = build_enriched_orders()
    assert enriched.count() <= fact_count, f"Enriched count ({enriched.count()}) exceeds fact count ({fact_count})"


def test_main_build_enriched_orders_creates_table():
    """After enriched_orders is built, the table should exist."""
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.enriched_orders"), "enriched_orders table does not exist after main build"


def test_enriched_orders_table_not_empty():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
    assert df.count() > 0, "enriched_orders table is empty"


def test_enriched_orders_no_duplicates():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
    total = df.count()
    distinct = df.dropDuplicates().count()
    assert total == distinct, f"Duplicates found in enriched_orders: {total} rows, {distinct} distinct"

In [0]:
# Apply schema and local build tests

def _make_orders_df(rows):
    schema = StructType([
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("product_id", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("discount", DoubleType(), True),
        StructField("profit", DoubleType(), True),
        StructField("quantity", IntegerType(), True),
    ])
    return spark.createDataFrame(rows, schema)


def test_dq_orders_profit_allows_negative():
    df = _make_orders_df([
        ("CA-2017-152156", "PW-19240", "FUR-CH-10002961", 100.5, 0.2, -25.68, 2)
    ])
    result = dq_orders(df)
    assert result.count() == 1, "Negative profit row should be retained"


def test_dq_orders_filters_negative_price():
    df = _make_orders_df([
        ("CA-2017-152156", "PW-19240", "FUR-CH-10002961", -100.5, 0.2, 25.68, 2)
    ])
    result = dq_orders(df)
    assert result.count() == 0, "Negative price row should be filtered"


def test_dq_orders_filters_negative_discount():
    df = _make_orders_df([
        ("CA-2017-152156", "PW-19240", "FUR-CH-10002961", 100.5, -0.2, 25.68, 2)
    ])
    result = dq_orders(df)
    assert result.count() == 0, "Negative discount row should be filtered"


def test_dq_orders_filters_zero_quantity():
    df = _make_orders_df([
        ("CA-2017-152156", "PW-19240", "FUR-CH-10002961", 100.5, 0.2, 25.68, 0)
    ])
    result = dq_orders(df)
    assert result.count() == 0, "Zero quantity row should be filtered"


def test_apply_order_schema_columns():
    df = dq_orders(stg_orders)
    df = dedupe(df)
    df = parse_dates(df, {"order_date": "d/M/yyyy", "ship_date": "d/M/yyyy"})
    result = apply_schema(df, order_schema)
    assert result.columns == [f.name for f in order_schema.fields]


def test_apply_schema_no_extra_columns():
    df = dq_orders(stg_orders)
    df = parse_dates(df, {"order_date": "d/M/yyyy", "ship_date": "d/M/yyyy"})
    result = apply_schema(df, order_schema)
    assert len(result.columns) == len(order_schema.fields)


def _build_fact_orders_temp():
    df = fill_defaults(stg_orders, configs["Orders"]["defaults"])
    df = dq_orders(df)
    df = dedupe(df)
    df = parse_dates(df, {"order_date": "d/M/yyyy", "ship_date": "d/M/yyyy"})
    df = apply_schema(df, order_schema)
    df = df.withColumn("order_key", F.monotonically_increasing_id())
    return df


def test_build_fact_orders_has_order_key():
    result = _build_fact_orders_temp()
    assert "order_key" in result.columns, "order_key column missing from local fact build"


def test_build_fact_orders_order_key_unique():
    result = _build_fact_orders_temp()
    total = result.count()
    distinct_keys = result.select("order_key").distinct().count()
    assert total == distinct_keys, f"order_key is not unique: {total} rows, {distinct_keys} distinct keys"




In [0]:
# Run all tests
def main_orders_test():
    test_functions = [
        # staging table
        test_stg_orders_exists,
        test_stg_orders_not_empty,
        # dq orders
        test_dq_orders_no_null_ids,
        test_dq_orders_no_blank_ids,
        test_dq_orders_order_id_format,
        test_dq_orders_customer_id_format,
        test_dq_orders_product_id_format,
        test_dq_orders_price_non_negative,
        test_dq_orders_discount_non_negative,
        test_dq_orders_quantity_positive_integer,
        test_dq_orders_reduces_or_keeps_count,
        test_dq_orders_profit_allows_negative,
        test_dq_orders_filters_negative_price,
        test_dq_orders_filters_negative_discount,
        test_dq_orders_filters_zero_quantity,
        # dedupe
        test_dedupe_orders_no_duplicates,
        test_dedupe_orders_reduces_or_keeps_count,
        test_dedupe_exact_duplicate_rows_removed,
        # schema and build
        test_apply_order_schema_columns,
        test_apply_schema_no_extra_columns,
        test_build_fact_orders_has_order_key,
        test_build_fact_orders_order_key_unique,
        # enriched orders
        test_dim_tables_exist,
        test_fact_orders_exists,
        test_build_enriched_orders_not_empty,
        test_build_enriched_orders_expected_columns,
        test_build_enriched_orders_no_customer_product_id_cols,
        test_build_enriched_orders_retains_fact_columns,
        test_build_enriched_orders_no_nulls_in_joined_cols,
        test_enriched_orders_count_le_fact,
        test_main_build_enriched_orders_creates_table,
        test_enriched_orders_table_not_empty,
        test_enriched_orders_no_duplicates,
    ]

    passed, failed = 0, 0
    for fn in test_functions:
        try:
            fn()
            passed += 1
            print(f"  [PASSED] {fn.__name__}")
        except Exception as e:
            failed += 1
            print(f"  [FAILED] {fn.__name__}: {e}")

    print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")



    # Cleanup
    spark.sql("DROP TABLE IF EXISTS az_ci_de_dbs.ecom_dev.fact_orders")
    spark.sql("DROP TABLE IF EXISTS az_ci_de_dbs.ecom_dev.enriched_orders")
    print("[CLEANUP] Dropped az_ci_de_dbs.ecom_dev.fact_orders")
    print("[CLEANUP] Dropped az_ci_de_dbs.ecom_dev.enriched_orders")